# Ad Creative Performance Predictor — Full Walkthrough

This notebook demonstrates the complete multimodal ML pipeline:

1. **Synthetic image generation** — simulated ad creatives
2. **Visual feature extraction** — colour, composition, ResNet embeddings
3. **Dataset EDA** — campaign metadata + performance distributions
4. **Model training** — multimodal GBDT fusion
5. **Evaluation** — CTR R², Conversion AUC, feature importance
6. **Prediction & explanation** — per-ad performance predictions
7. **SHAP-style feature importance** — what drives performance

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#080808',
    'axes.facecolor': '#080808',
    'axes.edgecolor': '#2A2A2A',
    'axes.labelcolor': '#CCC',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'grid.color': '#181818',
    'figure.dpi': 120,
    'font.family': 'monospace'
})

TIER_COLORS = {'top': '#C8F060', 'average': '#F0A060', 'poor': '#F06090'}
print('✓ Imports ready')

## 1. Ad Creative Images — Visual Feature Extraction

In [ ]:
from src.data_generator import generate_synthetic_ad_image
from src.visual_features import AdCreativeFeatureExtractor

# Generate a grid of synthetic ad creatives
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Synthetic Ad Creatives', color='#EEE', fontsize=12)

for i, ax in enumerate(axes.flat):
    img = generate_synthetic_ad_image(300, 250, seed=i * 7)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Ad #{i+1}', color='#888', fontsize=8)

plt.tight_layout()
plt.show()
print('Note: In production, these would be real ad creative images (PNG/JPG).')

In [ ]:
# Extract visual features from sample images
extractor = AdCreativeFeatureExtractor(load_backbone=True)

print('Extracting visual features from 10 sample ads...')
sample_features = []
for i in range(10):
    img = generate_synthetic_ad_image(300, 250, seed=i * 7)
    features = extractor.extract(img)
    sample_features.append(features)

print('\nSample Feature Extraction Results:')
print(f'{"Ad":<5} {"Temp":<8} {"Bright":<8} {"Sat":<7} {"Contrast":<10} {"Complexity":<12} {"Attention":<10} {"Category"}')
print('-' * 75)
for i, f in enumerate(sample_features):
    print(f'{i+1:<5} {f.colour_temperature:<8} {f.brightness:<8.3f} {f.saturation:<7.3f} '
          f'{f.contrast:<10.3f} {f.visual_complexity:<12.3f} {f.attention_score:<10.3f} {f.predicted_category}')

In [ ]:
# Visualise feature distributions across ads
feats_df = pd.DataFrame([
    {'brightness': f.brightness, 'saturation': f.saturation,
     'contrast': f.contrast, 'complexity': f.visual_complexity,
     'attention': f.attention_score, 'brand_safety': f.brand_safety_score,
     'text_ratio': f.text_area_ratio, 'rot_score': f.rule_of_thirds_score}
    for f in sample_features
])

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Visual Feature Distributions (10 Sample Ads)', color='#EEE', fontsize=12)

colors = ['#C8F060', '#60C8F0', '#F0A060', '#F06090', '#A060F0', '#60F0A0', '#F0F060', '#60A0F0']
for ax, col, color in zip(axes.flat, feats_df.columns, colors):
    ax.bar(range(len(feats_df)), feats_df[col], color=color, alpha=0.75)
    ax.set_title(col.replace('_', ' ').title(), color='#CCC', fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.grid(True, axis='y')

plt.tight_layout()
plt.show()

## 2. Campaign Dataset EDA

In [ ]:
from src.data_generator import generate_ad_dataset

df, vis_vectors, y_ctr, y_engagement, y_converted = generate_ad_dataset(n_ads=1000, seed=42)

print(f'Dataset: {len(df)} ads')
print(f'\nCTR stats: mean={y_ctr.mean():.4f}, std={y_ctr.std():.4f}, max={y_ctr.max():.4f}')
print(f'Engagement: mean={y_engagement.mean():.4f}, std={y_engagement.std():.4f}')
print(f'Conversion rate: {y_converted.mean():.1%}')
print(f'\nVisual feature matrix shape: {vis_vectors.shape}')
print(f'Structured feature matrix shape: {df.shape}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Campaign Dataset EDA', color='#EEE', fontsize=13)

# CTR distribution
axes[0,0].hist(y_ctr * 100, bins=40, color='#C8F060', alpha=0.75, edgecolor='#080808')
axes[0,0].axvline(y_ctr.mean() * 100, color='#F0A060', linewidth=2, label=f'Mean={y_ctr.mean()*100:.2f}%')
axes[0,0].set_title('CTR Distribution', color='#CCC', fontsize=10)
axes[0,0].set_xlabel('CTR (%)', color='#CCC')
axes[0,0].legend(fontsize=8, framealpha=0.3)
axes[0,0].grid(True)

# Engagement distribution
axes[0,1].hist(y_engagement * 100, bins=40, color='#60C8F0', alpha=0.75, edgecolor='#080808')
axes[0,1].axvline(y_engagement.mean() * 100, color='#F0A060', linewidth=2,
                  label=f'Mean={y_engagement.mean()*100:.2f}%')
axes[0,1].set_title('Engagement Rate Distribution', color='#CCC', fontsize=10)
axes[0,1].set_xlabel('Engagement Rate (%)', color='#CCC')
axes[0,1].legend(fontsize=8, framealpha=0.3)
axes[0,1].grid(True)

# CTR by ad format
format_labels = ['Banner', 'Video', 'Carousel', 'Story']
format_ctr = [y_ctr[df['ad_format'] == i].mean() for i in range(4)]
axes[0,2].bar(format_labels, [v * 100 for v in format_ctr],
              color=['#C8F060', '#60C8F0', '#F0A060', '#F06090'], alpha=0.8)
axes[0,2].set_title('Mean CTR by Ad Format', color='#CCC', fontsize=10)
axes[0,2].set_ylabel('Mean CTR (%)', color='#CCC')
axes[0,2].grid(True, axis='y')

# CTR vs budget
axes[1,0].scatter(df['budget_daily'], y_ctr * 100, alpha=0.3, s=8, color='#C8F060')
axes[1,0].set_title('CTR vs Daily Budget', color='#CCC', fontsize=10)
axes[1,0].set_xlabel('Daily Budget (£)', color='#CCC')
axes[1,0].set_ylabel('CTR (%)', color='#CCC')
axes[1,0].grid(True)

# CTA effect on CTR
ctr_no_cta = y_ctr[df['has_cta'] == 0]
ctr_cta    = y_ctr[df['has_cta'] == 1]
axes[1,1].boxplot([ctr_no_cta * 100, ctr_cta * 100],
                  labels=['No CTA', 'Has CTA'],
                  patch_artist=True,
                  boxprops=dict(facecolor='#1A1A1A', color='#60C8F0'),
                  medianprops=dict(color='#C8F060', linewidth=2),
                  whiskerprops=dict(color='#888'),
                  capprops=dict(color='#888'))
axes[1,1].set_title('CTR: CTA Effect', color='#CCC', fontsize=10)
axes[1,1].set_ylabel('CTR (%)', color='#CCC')
axes[1,1].grid(True, axis='y')

# Retargeting effect
ctr_no_ret = y_ctr[df['is_retargeting'] == 0]
ctr_ret    = y_ctr[df['is_retargeting'] == 1]
axes[1,2].boxplot([ctr_no_ret * 100, ctr_ret * 100],
                  labels=['Prospecting', 'Retargeting'],
                  patch_artist=True,
                  boxprops=dict(facecolor='#1A1A1A', color='#F0A060'),
                  medianprops=dict(color='#C8F060', linewidth=2),
                  whiskerprops=dict(color='#888'),
                  capprops=dict(color='#888'))
axes[1,2].set_title('CTR: Retargeting Effect', color='#CCC', fontsize=10)
axes[1,2].set_ylabel('CTR (%)', color='#CCC')
axes[1,2].grid(True, axis='y')

plt.tight_layout()
plt.show()

## 3. Model Training — Multimodal Fusion

In [ ]:
from src.performance_model import AdPerformanceModel

# Train/test split
n_train = 800
train_df = df.iloc[:n_train].reset_index(drop=True)
test_df  = df.iloc[n_train:].reset_index(drop=True)
train_vis, test_vis = vis_vectors[:n_train], vis_vectors[n_train:]
train_ctr, test_ctr = y_ctr[:n_train], y_ctr[n_train:]
train_eng, test_eng = y_engagement[:n_train], y_engagement[n_train:]
train_conv, test_conv = y_converted[:n_train], y_converted[n_train:]

# Fit model
model = AdPerformanceModel(
    use_visual_features=True,
    pca_components=20
)
model.fit(train_df, train_ctr, train_eng, train_conv, train_vis)

In [ ]:
# Evaluate on test set
metrics = model.evaluate(test_df, test_ctr, test_eng, test_conv, test_vis)

print('Model Evaluation Results')
print('='*40)
print(f'CTR R²:           {metrics.ctr_r2:.4f}')
print(f'CTR MAE:          {metrics.ctr_mae*100:.4f}%')
print(f'CTR MAPE:         {metrics.ctr_mape*100:.1f}%')
print(f'Engagement R²:    {metrics.engagement_r2:.4f}')
print(f'Engagement MAE:   {metrics.engagement_mae*100:.4f}%')
print(f'Conversion AUC:   {metrics.conversion_auc:.4f}')
print(f'CV R² (5-fold):   {metrics.cv_r2_mean:.4f} ± {metrics.cv_r2_std:.4f}')

In [ ]:
# Predicted vs Actual plots
test_results = model.predict(test_df, test_vis)
pred_ctr  = np.array([r.ctr for r in test_results])
pred_eng  = np.array([r.engagement_rate for r in test_results])
pred_conv = np.array([r.conversion_prob for r in test_results])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Predicted vs Actual — Test Set', color='#EEE', fontsize=13)

for ax, preds, actuals, title, color in [
    (axes[0], pred_ctr * 100,  test_ctr * 100,  f'CTR  (R²={metrics.ctr_r2:.3f})',        '#C8F060'),
    (axes[1], pred_eng * 100,  test_eng * 100,  f'Engagement  (R²={metrics.engagement_r2:.3f})', '#60C8F0'),
    (axes[2], pred_conv * 100, test_conv * 100, f'Conversion Prob vs Label (AUC={metrics.conversion_auc:.3f})', '#F0A060'),
]:
    ax.scatter(preds, actuals, alpha=0.35, s=10, color=color)
    mn, mx = min(preds.min(), actuals.min()), max(preds.max(), actuals.max())
    ax.plot([mn, mx], [mn, mx], color='#555', linewidth=1, linestyle='--')
    ax.set_xlabel('Predicted', color='#CCC')
    ax.set_ylabel('Actual', color='#CCC')
    ax.set_title(title, color='#CCC', fontsize=10)
    ax.grid(True)

plt.tight_layout()
plt.show()

## 4. Feature Importance

In [ ]:
importances = model._ctr_model.feature_importances_
feat_names  = model.feature_names_

# Top 15 features
top_n = 15
idx = np.argsort(importances)[-top_n:][::-1]
top_names = [feat_names[i] if i < len(feat_names) else f'feat_{i}' for i in idx]
top_vals  = importances[idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Feature Importance — CTR Model', color='#EEE', fontsize=13)

# Horizontal bar
colors = ['#C8F060' if 'visual' not in n else '#60C8F0' for n in top_names]
bars = axes[0].barh(range(top_n), top_vals[::-1], color=colors[::-1], alpha=0.8)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels([n.replace('_', ' ') for n in top_names[::-1]], color='#CCC', fontsize=9)
axes[0].set_xlabel('Feature Importance', color='#CCC')
axes[0].set_title('Top 15 Features', color='#CCC', fontsize=10)
axes[0].grid(True, axis='x')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#C8F060', alpha=0.8, label='Structured feature'),
    Patch(facecolor='#60C8F0', alpha=0.8, label='Visual (PCA) feature'),
]
axes[0].legend(handles=legend_elements, fontsize=8, framealpha=0.3)

# Cumulative importance
all_idx = np.argsort(importances)[::-1]
cumulative = np.cumsum(importances[all_idx])
axes[1].plot(range(1, len(cumulative)+1), cumulative * 100, color='#C8F060', linewidth=2)
axes[1].axhline(80, color='#F0A060', linestyle='--', linewidth=1, label='80% threshold')
n_80 = np.argmax(cumulative >= 0.8) + 1
axes[1].axvline(n_80, color='#F0A060', linestyle=':', linewidth=1)
axes[1].text(n_80 + 1, 40, f'{n_80} features\n→ 80% importance', color='#F0A060', fontsize=9)
axes[1].set_xlabel('Number of Features', color='#CCC')
axes[1].set_ylabel('Cumulative Importance (%)', color='#CCC')
axes[1].set_title('Cumulative Feature Importance', color='#CCC', fontsize=10)
axes[1].legend(fontsize=8, framealpha=0.3)
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5. Performance Tier Analysis

In [ ]:
tier_labels = [r.performance_tier for r in test_results]
tier_counts = pd.Series(tier_labels).value_counts()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Performance Tier Analysis', color='#EEE', fontsize=13)

# Tier distribution
tiers = ['top', 'average', 'poor']
counts = [tier_counts.get(t, 0) for t in tiers]
axes[0].bar(tiers, counts, color=[TIER_COLORS[t] for t in tiers], alpha=0.85)
for i, (t, c) in enumerate(zip(tiers, counts)):
    axes[0].text(i, c + 1, str(c), ha='center', color='#CCC', fontsize=10)
axes[0].set_title('Ads per Performance Tier', color='#CCC', fontsize=10)
axes[0].set_ylabel('Count', color='#CCC')
axes[0].grid(True, axis='y')

# CTR by tier
ctr_by_tier = {t: [test_ctr[i]*100 for i, tier in enumerate(tier_labels) if tier == t] for t in tiers}
axes[1].boxplot([ctr_by_tier[t] for t in tiers if ctr_by_tier[t]],
                labels=[t for t in tiers if ctr_by_tier[t]],
                patch_artist=True,
                boxprops=dict(facecolor='#1A1A1A'),
                medianprops=dict(linewidth=2),
                whiskerprops=dict(color='#888'),
                capprops=dict(color='#888'))
for patch, t in zip(axes[1].patches, [t for t in tiers if ctr_by_tier[t]]):
    patch.set_facecolor(TIER_COLORS[t] + '40')
    patch.set_edgecolor(TIER_COLORS[t])
axes[1].set_title('Actual CTR by Predicted Tier', color='#CCC', fontsize=10)
axes[1].set_ylabel('Actual CTR (%)', color='#CCC')
axes[1].grid(True, axis='y')

# Conversion rate by tier
conv_by_tier = {t: [test_conv[i] for i, tier in enumerate(tier_labels) if tier == t] for t in tiers}
conv_rates = [np.mean(conv_by_tier[t]) * 100 if conv_by_tier[t] else 0 for t in tiers]
axes[2].bar(tiers, conv_rates, color=[TIER_COLORS[t] for t in tiers], alpha=0.8)
for i, v in enumerate(conv_rates):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', color='#CCC', fontsize=9)
axes[2].set_title('Conversion Rate by Tier', color='#CCC', fontsize=10)
axes[2].set_ylabel('Conversion Rate (%)', color='#CCC')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.show()

## 6. Per-Ad Prediction & Explanation

In [ ]:
from IPython.display import Markdown

# Show predictions and explanations for 3 ads of different tiers
top_idx  = next((i for i, r in enumerate(test_results) if r.performance_tier == 'top'), 0)
avg_idx  = next((i for i, r in enumerate(test_results) if r.performance_tier == 'average'), 1)
poor_idx = next((i for i, r in enumerate(test_results) if r.performance_tier == 'poor'), 2)

for label, idx in [('TOP', top_idx), ('AVERAGE', avg_idx), ('POOR', poor_idx)]:
    r = test_results[idx]
    print(f'\n{"─"*60}')
    explanation = model.explain_prediction(r)
    display(Markdown(f'**[{label} AD — Test Ad #{idx}]**\n\n' + explanation))

## 7. Visual Feature Correlation with CTR

In [ ]:
# Extract scalar visual features from training images and correlate with CTR
print('Extracting visual features from 100 training ads for correlation analysis...')
visual_scalar_features = []
for i in range(min(100, n_train)):
    img = generate_synthetic_ad_image(200, 150, seed=i)
    feats = extractor.extract(img)
    visual_scalar_features.append({
        'brightness': feats.brightness,
        'saturation': feats.saturation,
        'contrast': feats.contrast,
        'complexity': feats.visual_complexity,
        'text_ratio': feats.text_area_ratio,
        'rot_score': feats.rule_of_thirds_score,
        'attention': feats.attention_score,
        'brand_safety': feats.brand_safety_score,
        'ctr': train_ctr[i] * 100
    })

vis_feat_df = pd.DataFrame(visual_scalar_features)
corr_with_ctr = vis_feat_df.corr()['ctr'].drop('ctr').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle('Visual Feature Correlation with CTR', color='#EEE', fontsize=12)
colors = ['#C8F060' if v >= 0 else '#F06090' for v in corr_with_ctr.values]
bars = ax.barh(corr_with_ctr.index, corr_with_ctr.values, color=colors, alpha=0.8)
ax.axvline(0, color='#555', linewidth=0.8)
for bar, val in zip(bars, corr_with_ctr.values):
    ax.text(val + (0.003 if val >= 0 else -0.003), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right',
            color='#CCC', fontsize=9)
ax.set_xlabel('Pearson Correlation with CTR', color='#CCC')
ax.grid(True, axis='x')

plt.tight_layout()
plt.show()
print('\nNote: In production with real creative data, correlation patterns would be stronger and more interpretable.')